# MambaVision Object Detection Notebook

This notebook is organized into: environment setup, dataset preparation, pretrained backbone loading, training, evaluation, and artifact export.

Run cells from top to bottom for a clean training flow.

# 1) Connect Google Drive

Mount Drive to access datasets, checkpoints, logs, and exported artifacts.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 2) Create Python 3.10 environment

Install Python 3.10 and build a dedicated virtual environment to match Torch/MMCV compatibility.

In [2]:
# ===== SYSTEM =====
!sudo apt-get update -q
!sudo apt-get install -y python3.10 python3.10-venv python3.10-dev

# ===== VENV =====
!python3.10 -m venv /content/py310
!/content/py310/bin/pip install --upgrade pip -q

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,533 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,970 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-bac

# 3) Install core dependencies

Install Torch, OpenMMLab stack, and Mamba-related libraries with pinned versions for reproducibility.

In [3]:
!/content/py310/bin/pip install -q \
  torch==2.1.0+cu121 torchvision==0.16.0+cu121 \
  --index-url https://download.pytorch.org/whl/cu121

!/content/py310/bin/pip install -q numpy==1.26.4
!/content/py310/bin/pip install -q opencv-python-headless==4.8.1.78

In [4]:
# ===== MM STACK =====
!/content/py310/bin/pip install -q \
  mmengine==0.10.1 \
  mmdet==3.3.0 \
  mmsegmentation==1.2.2 \
  mmpretrain==1.2.0 \
  pycocotools

In [5]:
!/content/py310/bin/pip install -q \
  mmcv==2.1.0 \
  -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html

In [6]:
# ===== OTHER LIBS =====
!/content/py310/bin/pip install -q \
  scikit-learn matplotlib timm transformers==4.37.2

# ===== MAMBA =====
!/content/py310/bin/pip install -q \
  https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4+cu12torch2.1cxx11abiFALSE-cp310-cp310-linux_x86_64.whl


In [7]:
!/content/py310/bin/pip uninstall -y numpy
!/content/py310/bin/pip install numpy==1.26.4

Found existing installation: numpy 2.2.6
Uninstalling numpy-2.2.6:
  Successfully uninstalled numpy-2.2.6
  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


In [8]:
# ===== CLONE REPO =====
!git clone https://github.com/NVlabs/MambaVision.git /content/MambaVision || true
%cd /content/MambaVision/object_detection

Cloning into '/content/MambaVision'...
remote: Enumerating objects: 783, done.
remote: Counting objects: 100% (221/221), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 783 (delta 183), reused 148 (delta 146), pack-reused 562 (from 1)
Receiving objects: 100% (783/783), 2.73 MiB | 18.79 MiB/s, done.
Resolving deltas: 100% (415/415), done.
/content/MambaVision/object_detection


# Load COCO Dataset

In [9]:
import os

COCO_ROOT = '/content/coco'
os.makedirs(COCO_ROOT, exist_ok=True)

if not os.path.exists(f"{COCO_ROOT}/annotations"):
    !wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O /tmp/ann.zip
    !unzip -q /tmp/ann.zip -d {COCO_ROOT}

if not os.path.exists(f"{COCO_ROOT}/val2017"):
    !wget -q http://images.cocodataset.org/zips/val2017.zip -O /tmp/val.zip
    !unzip -q /tmp/val.zip -d {COCO_ROOT}

In [10]:
import kagglehub, shutil
from pathlib import Path

path = kagglehub.dataset_download("trungit/coco25k")

SRC = Path(path) / "images"
DST = Path(COCO_ROOT) / "train2017"

if not DST.exists():
    shutil.copytree(SRC, DST)

Using Colab cache for faster access to the 'coco25k' dataset.


In [11]:
import json

with open(Path(path) / "coco25k.txt") as f:
    keep = set([line.strip().split("/")[-1] for line in f])

ann_path = Path(COCO_ROOT) / "annotations/instances_train2017.json"

with open(ann_path) as f:
    coco = json.load(f)

images = [img for img in coco["images"] if img["file_name"] in keep]
ids = {img["id"] for img in images}
anns = [a for a in coco["annotations"] if a["image_id"] in ids]

coco["images"] = images
coco["annotations"] = anns

with open(ann_path, "w") as f:
    json.dump(coco, f)

print("Filtered:", len(images))

Filtered: 25000


# Download pretrained backbone (MambaVision 1K)

Backbone load từ ImageNet-1K pretrain; detector heads (FPN, RPN, Cascade Mask R-CNN) train from scratch.

In [12]:
import os, shutil
from pathlib import Path

CKPT_DIR = "/content/ckpts"
os.makedirs(CKPT_DIR, exist_ok=True)

PRETRAINED_BACKBONE = f"{CKPT_DIR}/mambavision_tiny_1k.pth.tar"
DRIVE_BACKBONE = f"/content/drive/MyDrive/MambaVision/mambavision_tiny_1k.pth.tar"

if Path(DRIVE_BACKBONE).is_file() and not Path(PRETRAINED_BACKBONE).is_file():
    shutil.copy2(DRIVE_BACKBONE, PRETRAINED_BACKBONE)
    print("✔ Copied backbone from Drive")

elif not Path(PRETRAINED_BACKBONE).is_file():
    print("Downloading backbone...")
    !wget -q -O {PRETRAINED_BACKBONE} \
        https://huggingface.co/nvidia/MambaVision-T-1K/resolve/main/mambavision_tiny_1k.pth.tar

assert Path(PRETRAINED_BACKBONE).is_file()
print("✔ Backbone ready")

✔ Backbone ready


# Write training script

In [20]:
%%writefile train_object_detection.py
import argparse, glob, json, os, shutil, subprocess, sys, time, gc, numpy
from pathlib import Path

os.environ["MPLBACKEND"] = "agg"
import matplotlib.pyplot as plt

# ===== FIX CỨNG =====
CONFIG_FILE = "configs/mamba_vision/cascade_mask_rcnn_mamba_vision_tiny_3x_coco.py"
DEFAULT_BACKBONE = "/content/ckpts/mambavision_tiny_1k.pth.tar"

# ===== UTILS =====
def _gc_collect():
    gc.collect()

def run(cmd, cwd=None, on_tick=None, tick_seconds=600):
    print(f"\n[RUN] {cmd}\n")
    env = os.environ.copy()
    env["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"
    env["OMP_NUM_THREADS"] = "1"
    env["MKL_NUM_THREADS"] = "1"

    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env)
    last_tick = 0
    while True:
        if p.poll() is not None:
            break
        if on_tick and time.time() - last_tick > tick_seconds:
            try:
                on_tick()
            except Exception as e:
                print("[WARN]", e)
            last_tick = time.time()
        time.sleep(2)

    if p.returncode != 0:
        raise RuntimeError(cmd)

# ===== ARG =====
def parse_args():
    ap = argparse.ArgumentParser()
    ap.add_argument("--object_detection_root", default="/content/MambaVision/object_detection")
    ap.add_argument("--coco_root", default="/content/coco")
    ap.add_argument("--pretrained_backbone", default="")
    ap.add_argument("--load_from", default="")
    ap.add_argument("--work_dir", default="/content/ObjectDetection")
    ap.add_argument("--auto_resume", action="store_true")
    ap.add_argument("--drive_sync_dir", default="")
    ap.add_argument("--train_ann_file", default="annotations/instances_train2017.json")
    ap.add_argument("--max_epochs", type=int, default=6)
    ap.add_argument("--run_eval", action="store_true")
    return ap.parse_args()

# ===== CHECK =====
def assert_paths(cfg):
    assert Path(cfg["object_detection_root"]).is_dir()
    assert Path(cfg["coco_root"], "train2017").is_dir()
    assert Path(cfg["pretrained_backbone"]).is_file()

# ===== CKPT =====
def choose_resume_checkpoint(work_dir):
    p = Path(work_dir)
    if (p / "latest.pth").exists():
        return str(p / "latest.pth")
    iters = sorted(p.glob("iter_*.pth"))
    return str(iters[-1]) if iters else ""

def get_latest_iter_ckpt(work_dir):
    files = sorted(Path(work_dir).glob("iter_*.pth"))
    return files[-1] if files else None

def get_iter_from_checkpoint(ckpt_path):
    import torch
    ckpt = torch.load(ckpt_path, map_location="cpu")
    it = ckpt.get("meta", {}).get("iter", 0)
    del ckpt
    _gc_collect()
    return it

def merge_vis_data_to_training_log(work_dir, max_iter=None):
    vis_jsons = sorted(Path(work_dir).glob("*/vis_data/*.json"))
    if not vis_jsons:
        return

    src = vis_jsons[-1]
    dst = Path(work_dir) / "training_log.json"

    def _load_records(path):
        records = {}
        try:
            with open(path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.rstrip("\n")
                    if not line:
                        continue
                    try:
                        row = json.loads(line)
                    except json.JSONDecodeError:
                        continue
                    # (iter, mode) uniquely identifies a log row
                    key = (row.get("iter"), row.get("mode", ""))
                    records[key] = (row.get("iter", 0), line)
        except FileNotFoundError:
            pass
        return records

    existing = _load_records(dst)
    fresh    = _load_records(src)

    # vis_data wins over existing for same key
    merged = {**existing, **fresh}

    # Drop records beyond the resume checkpoint boundary
    if max_iter is not None:
        merged = {k: v for k, v in merged.items() if v[0] <= max_iter}

    if not merged:
        return

    # Sort by (iter asc, mode asc) → chronological
    sorted_lines = [
        line
        for _, (_, line) in sorted(
            merged.items(), key=lambda kv: (kv[1][0], kv[0][1] or "")
        )
    ]

    tmp = dst.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        f.write("\n".join(sorted_lines) + "\n")
    tmp.replace(dst)

    label = f" (capped at iter {max_iter})" if max_iter is not None else ""
    print(f"[OK] training_log.json rebuilt: {len(sorted_lines)} records{label}")

def append_vis_data(work_dir):
    vis_jsons = sorted(Path(work_dir).glob("*/vis_data/*.json"))
    if not vis_jsons:
        return

    src = vis_jsons[-1]
    dst = Path(work_dir) / "training_log.json"

    # ===== lấy last_iter từ file =====
    last_iter = -1
    if dst.is_file():
        try:
            with open(dst, "rb") as f:
                f.seek(-4096, 2)
                for line in reversed(f.readlines()):
                    try:
                        row = json.loads(line)
                        last_iter = row.get("iter", -1)
                        break
                    except:
                        continue
        except:
            pass

    new_lines = []

    with open(src, "r", encoding="utf-8") as f:
        for line in f:
            try:
                row = json.loads(line)
            except:
                continue

            if row.get("iter", -1) > last_iter:
                new_lines.append(line.rstrip("\n"))

    if not new_lines:
        return

    with open(dst, "a", encoding="utf-8") as f:
        for l in new_lines:
            f.write(l + "\n")

    print(f"[OK] Appended {len(new_lines)} log lines")

def wait_for_stable_file(path, wait=8):
    s1 = path.stat().st_size
    time.sleep(wait)
    s2 = path.stat().st_size
    return s1 == s2

def sync_live_artifacts(work_dir, out_dir=""):
    if not out_dir:
        return

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    ckpt = get_latest_iter_ckpt(work_dir)
    if ckpt is None:
        return

    iter_num = int(ckpt.stem.split("_")[1])

    last_iter = getattr(sync_live_artifacts, "last_iter", -1)
    if iter_num <= last_iter:
        return

    if not wait_for_stable_file(ckpt, 8):
        print("[WARN] checkpoint not stable, skip")
        return

    sync_live_artifacts.last_iter = iter_num

    # ===== copy checkpoint =====
    tmp = out / "latest.tmp.pth"
    final = out / "latest.pth"

    shutil.copy2(ckpt, tmp)
    os.replace(tmp, final)

    print(f"[OK] Synced checkpoint @ iter {iter_num}")

    # ===== append log nhẹ =====
    append_vis_data(work_dir)

    log_file = Path(work_dir) / "training_log.json"
    if log_file.is_file():
        shutil.copy2(log_file, out / "training_log.json")

    _gc_collect()

# ===== CFG =====
def build_cfg_options(cfg):
    coco = cfg["coco_root"].rstrip("/") + "/"

    opts = [
        f"data_root='{coco}'",

        # ===== DATASET =====
        f"train_dataloader.dataset.data_root='{coco}'",
        f"val_dataloader.dataset.data_root='{coco}'",
        f"test_dataloader.dataset.data_root='{coco}'",

        f"train_dataloader.dataset.ann_file='{cfg['train_ann_file']}'",
        f"val_dataloader.dataset.ann_file='annotations/instances_val2017.json'",
        f"test_dataloader.dataset.ann_file='annotations/instances_val2017.json'",

        # ===== QUAN TRỌNG (FIX LỖI) =====
        f"val_evaluator.ann_file='{coco}annotations/instances_val2017.json'",
        f"test_evaluator.ann_file='{coco}annotations/instances_val2017.json'",

        # ===== MODEL =====
        f"model.backbone.pretrained='{cfg['pretrained_backbone']}'",
        f"train_cfg.max_epochs={cfg['max_epochs']}",

        # ===== DATALOADER =====
        "train_dataloader.batch_size=2",
        "train_dataloader.num_workers=2",
        "train_dataloader.persistent_workers=False",

        "val_dataloader.batch_size=2",
        "val_dataloader.num_workers=2",
        "val_dataloader.persistent_workers=False",

        # ===== OPTIM =====
        "optim_wrapper.type=AmpOptimWrapper",
        "optim_wrapper.optimizer.type=AdamW",
        "optim_wrapper.optimizer.lr=2.5e-5",

        # ===== CHECKPOINT =====
        "default_hooks.checkpoint.by_epoch=False",
        "default_hooks.checkpoint.interval=1000",
        "default_hooks.checkpoint.max_keep_ckpts=5",
        "default_hooks.checkpoint.save_last=True",

        # ===== LOG =====
        "log_processor.by_epoch=False",
        "default_hooks.logger.interval=50",
    ]

    if cfg["load_from"]:
        opts += [f"load_from='{cfg['load_from']}'"]

    return " ".join(opts)

# ===== MAIN =====
def main():
    args = parse_args()

    pretrained = args.pretrained_backbone or DEFAULT_BACKBONE

    cfg = dict(
        object_detection_root=args.object_detection_root,
        coco_root=args.coco_root,
        pretrained_backbone=pretrained,
        load_from=args.load_from,
        work_dir=args.work_dir,
        train_ann_file=args.train_ann_file,
        max_epochs=args.max_epochs,
        drive_sync_dir=args.drive_sync_dir,
    )

    Path(cfg["work_dir"]).mkdir(parents=True, exist_ok=True)
    assert_paths(cfg)

    # ===== RESUME =====
    resume_ckpt = ""
    if args.auto_resume:

        # ---- load checkpoint từ Drive ----
        if cfg["drive_sync_dir"]:
            d = Path(cfg["drive_sync_dir"]) / "latest.pth"
            if d.exists():
                shutil.copy2(d, cfg["work_dir"])
                resume_ckpt = str(Path(cfg["work_dir"]) / "latest.pth")
                print("[INFO] Loaded checkpoint from Drive:", resume_ckpt)

        # ---- fallback local ----
        if not resume_ckpt:
            resume_ckpt = choose_resume_checkpoint(cfg["work_dir"])

        if resume_ckpt:
            print("[INFO] Resume:", resume_ckpt)
            resume_iter = get_iter_from_checkpoint(resume_ckpt)
            print(f"[INFO] Checkpoint iter: {resume_iter} — rebuilding log")
            merge_vis_data_to_training_log(cfg["work_dir"], max_iter=resume_iter)

        # ---- load log từ Drive nếu chưa có ----
        if cfg["drive_sync_dir"]:
            drive_log = Path(cfg["drive_sync_dir"]) / "training_log.json"
            local_log = Path(cfg["work_dir"]) / "training_log.json"

            if drive_log.is_file() and not local_log.exists():
                shutil.copy2(drive_log, local_log)
                print("[INFO] Loaded training_log.json from Drive")

    # ===== RUN =====
    cfg_opts = build_cfg_options(cfg)

    cmd = (
        f"PYTHONPATH=tools:$PYTHONPATH "
        f"{sys.executable} tools/train.py {CONFIG_FILE} "
        f"--work-dir {cfg['work_dir']} "
    )

    if resume_ckpt:
        cmd += f"--resume {resume_ckpt} "

    cmd += f"--cfg-options {cfg_opts}"

    run(
        cmd,
        cwd=cfg["object_detection_root"],
        on_tick=lambda: sync_live_artifacts(cfg["work_dir"], cfg["drive_sync_dir"]),
        tick_seconds=300,
    )

    sync_live_artifacts(cfg["work_dir"], cfg["drive_sync_dir"])
    print("[OK] Done")

if __name__ == "__main__":
    main()

Overwriting train_object_detection.py


# Train Object Detection

- **Backbone**: MambaVision (load pretrained 1K, frozen stages 0–1, fine-tune stages 2–3)
- **Detector**: Cascade Mask R-CNN + FPN — train **from scratch**
- **Schedule**: 3× (36 epochs, official), hoặc `t4_quick` (3 epochs) để debug nhanh
- **Dataset**: COCO 2017 (bbox + segm)

> Để chạy paper-aligned đầy đủ, dùng `--mode paper`. Để chạy thử nhanh, dùng `--mode t4_quick`.

In [14]:
# work_dir = "/content/ObjectDetection"

# #  XÓA sạch work_dir trước khi train
# if os.path.exists(work_dir):
#     print(f"[WARN] Removing old work_dir: {work_dir}")
#     shutil.rmtree(work_dir)

In [ ]:
!/content/py310/bin/python train_object_detection.py \
  --object_detection_root /content/MambaVision/object_detection \
  --coco_root /content/coco \
  --work_dir /content/ObjectDetection \
  --auto_resume \
  --drive_sync_dir /content/drive/MyDrive/ObjectDetection \
  --max_epochs 6


[INFO] Loaded checkpoint from Drive: /content/ObjectDetection/latest.pth
[INFO] Resume: /content/ObjectDetection/latest.pth
[INFO] Checkpoint iter: 30500 — rebuilding log
[OK] training_log.json rebuilt: 608 records (capped at iter 30500)

[RUN] PYTHONPATH=tools:$PYTHONPATH /content/py310/bin/python tools/train.py configs/mamba_vision/cascade_mask_rcnn_mamba_vision_tiny_3x_coco.py --work-dir /content/ObjectDetection --resume /content/ObjectDetection/latest.pth --cfg-options data_root='/content/coco/' train_dataloader.dataset.data_root='/content/coco/' val_dataloader.dataset.data_root='/content/coco/' test_dataloader.dataset.data_root='/content/coco/' train_dataloader.dataset.ann_file='annotations/instances_train2017.json' val_dataloader.dataset.ann_file='annotations/instances_val2017.json' test_dataloader.dataset.ann_file='annotations/instances_val2017.json' val_evaluator.ann_file='/content/coco/annotations/instances_val2017.json' test_evaluator.ann_file='/content/coco/annotations/insta

\

# Write eval script

In [16]:
%%writefile eval_object_detection.py
import argparse, glob, json, os, subprocess, sys
from pathlib import Path

# ===== FIX CỨNG =====
CONFIG_FILE = "configs/mamba_vision/cascade_mask_rcnn_mamba_vision_tiny_3x_coco.py"
DEFAULT_BACKBONE = "/content/drive/MyDrive/MambaVision/mambavision_tiny_1k.pth.tar"

# ===== RUN =====
def run(cmd, cwd=None):
    env = os.environ.copy()
    env["MPLBACKEND"] = "Agg"

    print(f"\n[RUN] {cmd}\n")
    p = subprocess.run(cmd, shell=True, cwd=cwd, env=env)

    if p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode})")


# ===== CHECKPOINT =====
def choose_checkpoint(work_dir):
    p = Path(work_dir)

    # ưu tiên best đúng nghĩa
    best = sorted(p.glob("best*.pth"))
    if best:
        return str(best[-1])

    latest = p / "latest.pth"
    if latest.exists():
        return str(latest)

    iters = sorted(p.glob("iter_*.pth"))
    if iters:
        return str(iters[-1])

    return ""


# ===== LOG =====
def parse_log(path):
    bbox, segm = {}, {}

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                row = json.loads(line)
            except:
                continue

            if row.get("mode") != "val":
                continue

            ep = row.get("epoch")
            if ep is None:
                continue

            if "coco/bbox_mAP" in row:
                bbox[ep] = row["coco/bbox_mAP"]

            if "coco/segm_mAP" in row:
                segm[ep] = row["coco/segm_mAP"]

    return bbox, segm


def report(work_dir):
    logs = sorted(glob.glob(str(Path(work_dir) / "*.json")))
    if not logs:
        print("[WARN] No log found")
        return

    bbox, segm = parse_log(logs[-1])

    if not bbox:
        print("[WARN] No validation data")
        return

    last_ep = max(bbox.keys())

    print("=" * 50)
    print("Training Report")
    print("=" * 50)
    print(f"Epoch        : {last_ep}")
    print(f"BBox mAP     : {bbox[last_ep]:.4f}")

    if last_ep in segm:
        print(f"Segm mAP     : {segm[last_ep]:.4f}")

    print("=" * 50)


# ===== MAIN =====
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--object_detection_root", default="/content/MambaVision/object_detection")
    ap.add_argument("--coco_root", default="/content/coco")
    ap.add_argument("--work_dir", default="/content/ObjectDetection")
    ap.add_argument("--ckpt", default="")
    ap.add_argument("--pretrained_backbone", default="")
    args = ap.parse_args()

    pretrained = args.pretrained_backbone or DEFAULT_BACKBONE

    # ===== REPORT =====
    report(args.work_dir)

    # ===== CHECKPOINT =====
    ckpt = args.ckpt or choose_checkpoint(args.work_dir)
    assert ckpt and Path(ckpt).exists(), f"No checkpoint found: {args.work_dir}"

    print(f"\n[INFO] Evaluating: {ckpt}")

    coco = args.coco_root.rstrip("/") + "/"

    cfg_opts = " ".join([
        f"data_root='{coco}'",
        f"val_dataloader.dataset.data_root='{coco}'",
        f"test_dataloader.dataset.data_root='{coco}'",

        f"val_evaluator.ann_file='{coco}annotations/instances_val2017.json'",
        f"test_evaluator.ann_file='{coco}annotations/instances_val2017.json'",

        f"model.backbone.pretrained='{pretrained}'",

        "val_dataloader.batch_size=4",
        "val_dataloader.num_workers=4",
        "val_dataloader.persistent_workers=False",

        "test_dataloader.batch_size=4",
        "test_dataloader.num_workers=4",
        "test_dataloader.persistent_workers=False",
    ])

    cmd = (
        f"PYTHONPATH=tools:$PYTHONPATH "
        f"{sys.executable} tools/test.py {CONFIG_FILE} {ckpt} "
        f"--cfg-options {cfg_opts}"
    )

    run(cmd, cwd=args.object_detection_root)


if __name__ == "__main__":
    main()

Writing eval_object_detection.py


# Evaluate (bbox + segm mAP)

Chạy `eval_object_detection.py`:
1. In report nhanh từ training log (bbox/segm mAP)
2. Chạy `tools/test.py --eval bbox segm` chính thức của MMDet

In [17]:
!/content/py310/bin/python eval_object_detection.py \
    --coco_root /content/coco \
    --pretrained_backbone /content/ckpts/mambavision_tiny_1k.pth.tar \
    --work_dir /content/drive/MyDrive/ObjectDetection \


[WARN] No JSON log found
Traceback (most recent call last):
  File "/content/MambaVision/object_detection/eval_object_detection.py", line 167, in <module>
    main()
  File "/content/MambaVision/object_detection/eval_object_detection.py", line 132, in main
    assert ckpt, f"No checkpoint found in {work_dir}"
AssertionError: No checkpoint found in {WORK_DIR}


# Training Curves

In [18]:
from pathlib import Path
from IPython.display import Image as IPImage, display

chart_path = str(Path(WORK_DIR) / 'object_detection_training_curve.png')
if Path(chart_path).is_file():
    display(IPImage(chart_path))
else:
    print(f'[WARN] Chart not found: {chart_path}\nRun training first.')

[WARN] Chart not found: /content/drive/MyDrive/ObjectDetection/object_detection_training_curve.png
Run training first.


# Report final metrics (from checkpoint log)

---



In [ ]:
%%writefile report_det_metrics.py
import glob
import json
import os
from pathlib import Path


def main():
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--work_dir", type=str, required=True)
    args = parser.parse_args()

    json_logs = sorted(glob.glob(str(Path(args.work_dir) / "*.json")))
    if not json_logs:
        print("[ERROR] No JSON log found.")
        return

    rows_by_epoch = {}
    with open(json_logs[-1], encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue
            epoch = row.get("epoch")
            if epoch is None:
                continue
            if epoch not in rows_by_epoch:
                rows_by_epoch[epoch] = {}
            rows_by_epoch[epoch].update(row)

    epochs = sorted(rows_by_epoch.keys())
    if not epochs:
        print("[ERROR] No epoch data found.")
        return

    print("=" * 80)
    print("MambaVision Object Detection — Final Metrics Report")
    print("=" * 80)
    last = rows_by_epoch[epochs[-1]]
    metrics = [
        ("Epochs trained",           len(epochs)),
        ("Val bbox mAP",             last.get("coco/bbox_mAP",    "N/A")),
        ("Val bbox mAP_50",          last.get("coco/bbox_mAP_50", "N/A")),
        ("Val bbox mAP_75",          last.get("coco/bbox_mAP_75", "N/A")),
        ("Val bbox mAP_s",           last.get("coco/bbox_mAP_s",  "N/A")),
        ("Val bbox mAP_m",           last.get("coco/bbox_mAP_m",  "N/A")),
        ("Val bbox mAP_l",           last.get("coco/bbox_mAP_l",  "N/A")),
        ("Val segm mAP",             last.get("coco/segm_mAP",    "N/A")),
        ("Val segm mAP_50",          last.get("coco/segm_mAP_50", "N/A")),
        ("Val segm mAP_75",          last.get("coco/segm_mAP_75", "N/A")),
        ("Final train loss",         last.get("loss",             "N/A")),
    ]
    for name, val in metrics:
        if isinstance(val, float):
            print(f"  {name:<30}: {val:.4f}")
        else:
            print(f"  {name:<30}: {val}")
    print("=" * 80)


if __name__ == "__main__":
    main()

In [ ]:
!/content/py310/bin/python report_det_metrics.py --work_dir {WORK_DIR}